In [1]:
!nvidia-smi
!python --version

Wed Apr 15 22:42:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip -q uninstall -y torch torchvision torchaudio vllm
!pip -q install -U uv
!uv pip install vllm --torch-backend=auto
!pip -q install transformers accelerate bitsandbytes fastapi uvicorn psutil GPUtil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 111.3 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 2.53s
Prepared 73 packages in 38.21s
Uninstalled 8 packages in 437ms
Installed 73 packages in 1.38s
 + anthropic==0.95.0
 + apache-tvm-ffi==0.1.10
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==5.9.0
 + compressed-tensors==0.14.0.1
 - cuda-bindings==12.9.4
 + cuda-bindings==13.0.3
 - cuda-python==12.9.4
 + cuda-python==13.0.3
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.17.0
 + fastar==0.11.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 + gguf==0.18.0
 - huggingface-hub==1.10.1
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mistral-common==1.11.0
 + model-hosting-container-sta

In [3]:
!python - <<'PY'
import torch, vllm, transformers
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("vLLM:", vllm.__version__)
print("Transformers:", transformers.__version__)


/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
PyTorch: 2.10.0+cu130
CUDA available: True
GPU count: 1
GPU: NVIDIA A100-SXM4-40GB
vLLM: 0.19.0
Transformers: 4.57.6


In [5]:
!nohup vllm serve Qwen/Qwen3-8B \
  --dtype bfloat16 \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.9 \
  > vllm.log 2>&1 &

In [6]:
!tail -n 50 vllm.log

(EngineCore pid=4842) INFO 04-15 22:50:32 [monitor.py:48] torch.compile took 6.75 s in total
(EngineCore pid=4842) INFO 04-15 22:50:32 [monitor.py:76] Initial profiling/warmup run took 0.17 s
(EngineCore pid=4842) INFO 04-15 22:50:43 [kv_cache_utils.py:829] Overriding num_gpu_blocks=0 with num_gpu_blocks_override=512
(EngineCore pid=4842) INFO 04-15 22:50:43 [gpu_model_runner.py:5876] Profiling CUDA graph memory: PIECEWISE=51 (largest=512), FULL=35 (largest=256)
(EngineCore pid=4842) INFO 04-15 22:50:45 [gpu_model_runner.py:5955] Estimated CUDA graph memory: 0.52 GiB total
(EngineCore pid=4842) INFO 04-15 22:50:45 [gpu_worker.py:436] Available KV cache memory: 19.54 GiB
(EngineCore pid=4842) INFO 04-15 22:50:45 [gpu_worker.py:470] In v0.19, CUDA graph memory profiling will be enabled by default (VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1), which more accurately accounts for CUDA graph memory during KV cache allocation. To try it now, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1 and i

In [7]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen3-8B","object":"model","created":1776293568,"owned_by":"vllm","root":"Qwen/Qwen3-8B","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-ae99f724cb32cba5","object":"model_permission","created":1776293568,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [11]:
import requests

url = "http://localhost:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen3-8B",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write 3 short bullet points about A100 GPUs."}
    ],
    "temperature": 0.7,
    "max_tokens": 128
}

try:
    resp = requests.post(url, json=payload, timeout=60)
    print(resp.status_code)
    print(resp.json())
except Exception as e:
    print("ERROR:", e)

200
{'id': 'chatcmpl-9ff911026e7afb0c', 'object': 'chat.completion', 'created': 1776293659, 'model': 'Qwen/Qwen3-8B', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "<think>\nOkay, the user wants three short bullet points about A100 GPUs. Let me start by recalling what I know about the A100. It's part of NVIDIA's A series, right? So first, I should mention its purpose. The A100 is designed for AI and high-performance computing. That's a key point.\n\nNext, the architecture. I think it uses the Ampere architecture, which is important for performance. Also, it has a large number of CUDA cores. Maybe around 54 billion? Wait, no, that's the total number of transistors. Let me check", 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning': None}, 'logprobs': None, 'finish_reason': 'length', 'stop_reason': None, 'token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 32, 'total_

In [9]:
print(response)

<Response [200]>
